# Reprodução do Experimento SOME/IP IDS — CTGAN + XGBoost

Notebook preparado para Google Colab.

Estrutura esperada:

```text
/content/datasets/
  X_train.npy
  y_train.npy
  X_test.npy
  y_test.npy
```

Objetivo:

1. Validar distribuição do dataset.
2. Criar teste balanceado por downsampling.
3. Treinar baseline XGBoost sem CTGAN.
4. Treinar CTGAN apenas com ataques do treino.
5. Gerar ataques sintéticos até balancear o treino.
6. Treinar XGBoost com hiperparâmetros do artigo.
7. Avaliar em cenário realista desbalanceado e cenário controlado balanceado.

Referência de parâmetros do artigo:

- XGBoost: 1000 árvores, learning rate 0.05, max depth 6, subsample 0.8, colsample 0.8, min child weight 1, lambda 1, gamma 0.
- CTGAN: embedding_dim 128, generator/critic 256/256, batch_size 500, pac 10, epochs 100.


## 0. Instalação das dependências

No Colab, rode esta célula primeiro. Se usar GPU, vá em **Runtime > Change runtime type > GPU**.


In [ ]:
!pip -q install xgboost scikit-learn pandas numpy matplotlib sdv ctgan joblib pyarrow

## 1. Imports e configuração global

In [ ]:
import os
import gc
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)
from sklearn.utils import resample

import xgboost as xgb
import joblib

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
DATA_DIR = Path('/content/datasets')
OUT_DIR = Path('/content/outputs_someip_xgboost')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR:', DATA_DIR)
print('OUT_DIR:', OUT_DIR)


## 2. Parâmetros de execução

Para reprodução completa, deixe `RUN_FULL_CTGAN = True` e `CTGAN_EPOCHS = 100`.

Para validar rapidamente o pipeline, use `QUICK_MODE = True`.


In [ ]:
# Modo rápido para testar se tudo funciona antes do treino completo.
QUICK_MODE = False

# CTGAN completo pode demorar bastante. Em Colab gratuito pode ser pesado.
RUN_FULL_CTGAN = True

# Parâmetros do artigo
CTGAN_EPOCHS = 100
CTGAN_BATCH_SIZE = 500
CTGAN_EMBEDDING_DIM = 128
CTGAN_GENERATOR_DIM = (256, 256)
CTGAN_DISCRIMINATOR_DIM = (256, 256)
CTGAN_PAC = 10

# Se QUICK_MODE=True, limita amostras para teste do notebook
QUICK_TRAIN_SAMPLES = 300_000
QUICK_TEST_SAMPLES = 300_000
QUICK_ATTACK_CTGAN_SAMPLES = 100_000
QUICK_CTGAN_EPOCHS = 5

# Para economizar RAM, o dataset aumentado pode ser salvo em parquet/npz em vez de ficar tudo em memória.
SAVE_SYNTHETIC_ATTACKS = True
SYNTHETIC_PATH = OUT_DIR / 'synthetic_attacks.npy'

# XGBoost do artigo
XGB_PARAMS = dict(
    objective='binary:logistic',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=1,
    reg_lambda=1.0,
    gamma=0.0,
    eval_metric='aucpr',
    tree_method='hist',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

print(json.dumps({
    'QUICK_MODE': QUICK_MODE,
    'RUN_FULL_CTGAN': RUN_FULL_CTGAN,
    'CTGAN_EPOCHS': CTGAN_EPOCHS,
    'XGB_PARAMS': XGB_PARAMS,
}, indent=2, default=str))


## 3. Carregamento dos `.npy`

Usa `mmap_mode='r'` para reduzir RAM durante inspeção. Algumas etapas precisam materializar arrays em memória.

In [ ]:
required_files = ['X_train.npy', 'y_train.npy', 'X_test.npy', 'y_test.npy']
for f in required_files:
    path = DATA_DIR / f
    if not path.exists():
        raise FileNotFoundError(f'Arquivo não encontrado: {path}')

X_train = np.load(DATA_DIR / 'X_train.npy', mmap_mode='r')
y_train = np.load(DATA_DIR / 'y_train.npy', mmap_mode='r')
X_test = np.load(DATA_DIR / 'X_test.npy', mmap_mode='r')
y_test = np.load(DATA_DIR / 'y_test.npy', mmap_mode='r')

print('X_train:', X_train.shape, X_train.dtype)
print('y_train:', y_train.shape, y_train.dtype)
print('X_test :', X_test.shape, X_test.dtype)
print('y_test :', y_test.shape, y_test.dtype)


## 4. Validação da distribuição de classes

In [ ]:
def class_counts(y):
    vals, counts = np.unique(np.asarray(y), return_counts=True)
    return dict(zip(vals.astype(int).tolist(), counts.astype(int).tolist()))

train_counts = class_counts(y_train)
test_counts = class_counts(y_test)

print('Train counts:', train_counts)
print('Test counts :', test_counts)

summary = pd.DataFrame([
    {'split': 'train_original', 'normal_0': train_counts.get(0, 0), 'attack_1': train_counts.get(1, 0), 'total': len(y_train)},
    {'split': 'test_imbalanced', 'normal_0': test_counts.get(0, 0), 'attack_1': test_counts.get(1, 0), 'total': len(y_test)},
])
summary['attack_pct'] = summary['attack_1'] / summary['total'] * 100
summary['normal_pct'] = summary['normal_0'] / summary['total'] * 100
summary


## 5. Preparar subconjuntos caso `QUICK_MODE=True`

In [ ]:
def stratified_indices(y, n_total, random_state=42):
    rng = np.random.default_rng(random_state)
    y_arr = np.asarray(y)
    classes, counts = np.unique(y_arr, return_counts=True)
    idx_all = []
    for c, count in zip(classes, counts):
        cls_idx = np.where(y_arr == c)[0]
        n_c = max(1, int(round(n_total * count / len(y_arr))))
        n_c = min(n_c, len(cls_idx))
        idx_all.append(rng.choice(cls_idx, size=n_c, replace=False))
    idx = np.concatenate(idx_all)
    rng.shuffle(idx)
    return idx

if QUICK_MODE:
    train_idx = stratified_indices(y_train, QUICK_TRAIN_SAMPLES, RANDOM_STATE)
    test_idx = stratified_indices(y_test, QUICK_TEST_SAMPLES, RANDOM_STATE)
    X_train_work = np.asarray(X_train[train_idx], dtype=np.float32)
    y_train_work = np.asarray(y_train[train_idx], dtype=np.int8)
    X_test_work = np.asarray(X_test[test_idx], dtype=np.float32)
    y_test_work = np.asarray(y_test[test_idx], dtype=np.int8)
else:
    # Cuidado: materializar tudo em RAM. Necessário para treino completo do XGBoost/sklearn.
    X_train_work = np.asarray(X_train, dtype=np.float32)
    y_train_work = np.asarray(y_train, dtype=np.int8)
    X_test_work = np.asarray(X_test, dtype=np.float32)
    y_test_work = np.asarray(y_test, dtype=np.int8)

print('Work train:', X_train_work.shape, class_counts(y_train_work))
print('Work test :', X_test_work.shape, class_counts(y_test_work))


## 6. Criar teste balanceado por downsampling

Cenário B do artigo/slide: mesma quantidade de normais e ataques no teste.

In [ ]:
def make_balanced_test(X, y, random_state=42):
    rng = np.random.default_rng(random_state)
    y = np.asarray(y)
    normal_idx = np.where(y == 0)[0]
    attack_idx = np.where(y == 1)[0]
    n = min(len(normal_idx), len(attack_idx))
    normal_down = rng.choice(normal_idx, size=n, replace=False)
    attack_down = rng.choice(attack_idx, size=n, replace=False)
    idx = np.concatenate([normal_down, attack_down])
    rng.shuffle(idx)
    return X[idx], y[idx], idx

X_test_bal, y_test_bal, test_bal_idx = make_balanced_test(X_test_work, y_test_work, RANDOM_STATE)
print('Balanced test:', X_test_bal.shape, class_counts(y_test_bal))


## 7. Funções de avaliação e threshold ótimo

O artigo menciona seleção de threshold maximizando F1, em vez de usar 0.5 fixo.

In [ ]:
def find_best_threshold_by_f1(y_true, y_score):
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    # thresholds tem tamanho N-1; precision/recall têm N
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    best_idx = int(np.nanargmax(f1))
    return float(thresholds[best_idx]), float(f1[best_idx]), float(precision[best_idx]), float(recall[best_idx])


def evaluate_scores(y_true, y_score, threshold=None, label='eval'):
    pr_auc = average_precision_score(y_true, y_score)
    roc_auc = roc_auc_score(y_true, y_score)
    if threshold is None:
        threshold, best_f1, best_p, best_r = find_best_threshold_by_f1(y_true, y_score)
    y_pred = (y_score >= threshold).astype(np.int8)
    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    out = {
        'label': label,
        'threshold': float(threshold),
        'pr_auc': float(pr_auc),
        'roc_auc': float(roc_auc),
        'f1': float(f1),
        'precision': float(precision),
        'recall': float(recall),
        'tn': int(cm[0,0]),
        'fp': int(cm[0,1]),
        'fn': int(cm[1,0]),
        'tp': int(cm[1,1]),
    }
    return out


def print_eval(result):
    print(json.dumps(result, indent=2))


## 8. Baseline: XGBoost sem CTGAN

Este experimento não é o resultado final do artigo, mas serve como controle para medir o ganho do CTGAN.

In [ ]:
start = time.time()

xgb_baseline = xgb.XGBClassifier(**XGB_PARAMS)
xgb_baseline.fit(X_train_work, y_train_work)

score_train_base = xgb_baseline.predict_proba(X_train_work)[:, 1]
threshold_base, f1_base_train, p_base_train, r_base_train = find_best_threshold_by_f1(y_train_work, score_train_base)
print('Baseline threshold from train:', threshold_base, 'train F1:', f1_base_train)

gc.collect()

score_test_imb_base = xgb_baseline.predict_proba(X_test_work)[:, 1]
score_test_bal_base = xgb_baseline.predict_proba(X_test_bal)[:, 1]

baseline_results = [
    evaluate_scores(y_test_work, score_test_imb_base, threshold_base, 'baseline_test_imbalanced'),
    evaluate_scores(y_test_bal, score_test_bal_base, threshold_base, 'baseline_test_balanced'),
]

for r in baseline_results:
    print_eval(r)

joblib.dump(xgb_baseline, OUT_DIR / 'xgb_baseline_no_ctgan.joblib')
pd.DataFrame(baseline_results).to_csv(OUT_DIR / 'baseline_results.csv', index=False)

print('Tempo baseline min:', (time.time() - start) / 60)


## 9. Treinar CTGAN somente com ataques do treino

Atenção: essa é a parte mais pesada do notebook. No Colab gratuito pode demorar ou estourar RAM.

Se quiser só validar o pipeline, configure `QUICK_MODE=True` e `QUICK_CTGAN_EPOCHS=5` no início.

In [ ]:
if RUN_FULL_CTGAN:
    try:
        from ctgan import CTGAN
    except Exception as e:
        raise RuntimeError('Não foi possível importar CTGAN. Verifique a instalação do pacote ctgan.') from e

    attack_idx = np.where(y_train_work == 1)[0]
    X_attack = X_train_work[attack_idx]

    if QUICK_MODE and len(X_attack) > QUICK_ATTACK_CTGAN_SAMPLES:
        rng = np.random.default_rng(RANDOM_STATE)
        sel = rng.choice(np.arange(len(X_attack)), size=QUICK_ATTACK_CTGAN_SAMPLES, replace=False)
        X_attack_ctgan = X_attack[sel]
        epochs = QUICK_CTGAN_EPOCHS
    else:
        X_attack_ctgan = X_attack
        epochs = CTGAN_EPOCHS

    feature_cols = [f'f{i}' for i in range(X_attack_ctgan.shape[1])]
    attack_df = pd.DataFrame(np.asarray(X_attack_ctgan, dtype=np.float32), columns=feature_cols)

    print('CTGAN train attack_df:', attack_df.shape)
    print('epochs:', epochs)

    start = time.time()
    ctgan = CTGAN(
        embedding_dim=CTGAN_EMBEDDING_DIM,
        generator_dim=CTGAN_GENERATOR_DIM,
        discriminator_dim=CTGAN_DISCRIMINATOR_DIM,
        batch_size=CTGAN_BATCH_SIZE,
        epochs=epochs,
        pac=CTGAN_PAC,
        verbose=True,
        cuda=True,
    )
    ctgan.fit(attack_df, discrete_columns=[])

    joblib.dump(ctgan, OUT_DIR / 'ctgan_attack_only.joblib')
    print('Tempo CTGAN min:', (time.time() - start) / 60)
else:
    print('RUN_FULL_CTGAN=False: pulando treino CTGAN')


## 10. Gerar ataques sintéticos para balancear o treino

No artigo, o treino aumentado fica aproximadamente:

- normal: 6.285.515
- ataque real + sintético: 7.116.673
- total: 13.402.188

Aqui geramos sintéticos suficientes para que a quantidade final de ataques se aproxime do número de normais ou, opcionalmente, do total descrito no artigo.

In [ ]:
if RUN_FULL_CTGAN:
    n_normal_train = int(np.sum(y_train_work == 0))
    n_attack_real = int(np.sum(y_train_work == 1))

    # Para balanceamento clássico: gerar até ataque_final == normal
    n_synth_needed = max(0, n_normal_train - n_attack_real)

    # Para reproduzir literalmente a tabela do artigo: ataque_final ~= 7,116,673.
    # Se quiser usar esse modo, descomente a linha abaixo:
    # n_synth_needed = max(0, len(y_train_work) - n_attack_real - 1)

    if QUICK_MODE:
        n_synth_needed = min(n_synth_needed, QUICK_TRAIN_SAMPLES)

    print('normal_train:', n_normal_train)
    print('attack_real:', n_attack_real)
    print('synthetic_needed:', n_synth_needed)

    start = time.time()
    synth_df = ctgan.sample(n_synth_needed)
    X_synth_attack = synth_df.values.astype(np.float32)

    # Opcional: clipping para intervalo observado nos ataques reais, reduzindo valores absurdos do GAN.
    mins = np.min(X_attack, axis=0)
    maxs = np.max(X_attack, axis=0)
    X_synth_attack = np.clip(X_synth_attack, mins, maxs).astype(np.float32)

    if SAVE_SYNTHETIC_ATTACKS:
        np.save(SYNTHETIC_PATH, X_synth_attack)
        print('Synthetic saved:', SYNTHETIC_PATH)

    print('X_synth_attack:', X_synth_attack.shape)
    print('Tempo geração sintéticos min:', (time.time() - start) / 60)
else:
    print('RUN_FULL_CTGAN=False: pulando geração sintética')


## 11. Montar treino aumentado CTGAN + XGBoost final

In [ ]:
if RUN_FULL_CTGAN:
    X_train_aug = np.vstack([
        X_train_work,
        X_synth_attack,
    ]).astype(np.float32)

    y_train_aug = np.concatenate([
        y_train_work,
        np.ones(len(X_synth_attack), dtype=np.int8),
    ])

    # Embaralha treino aumentado
    rng = np.random.default_rng(RANDOM_STATE)
    perm = rng.permutation(len(y_train_aug))
    X_train_aug = X_train_aug[perm]
    y_train_aug = y_train_aug[perm]

    print('Aug train:', X_train_aug.shape, class_counts(y_train_aug))
else:
    print('RUN_FULL_CTGAN=False')


## 12. Treinar XGBoost final com CTGAN

In [ ]:
if RUN_FULL_CTGAN:
    start = time.time()

    xgb_ctgan = xgb.XGBClassifier(**XGB_PARAMS)
    xgb_ctgan.fit(X_train_aug, y_train_aug)

    joblib.dump(xgb_ctgan, OUT_DIR / 'xgb_ctgan_augmented.joblib')
    print('Tempo XGBoost CTGAN min:', (time.time() - start) / 60)
else:
    print('RUN_FULL_CTGAN=False')


## 13. Threshold ótimo no treino aumentado

In [ ]:
if RUN_FULL_CTGAN:
    start = time.time()
    score_train_aug = xgb_ctgan.predict_proba(X_train_aug)[:, 1]
    threshold_ctgan, f1_train_ctgan, p_train_ctgan, r_train_ctgan = find_best_threshold_by_f1(y_train_aug, score_train_aug)
    print('threshold_ctgan:', threshold_ctgan)
    print('train_aug F1:', f1_train_ctgan, 'precision:', p_train_ctgan, 'recall:', r_train_ctgan)
    print('Tempo threshold min:', (time.time() - start) / 60)
else:
    print('RUN_FULL_CTGAN=False')


## 14. Avaliação final: cenário A desbalanceado e cenário B balanceado

In [ ]:
if RUN_FULL_CTGAN:
    start = time.time()

    score_test_imb = xgb_ctgan.predict_proba(X_test_work)[:, 1]
    score_test_bal = xgb_ctgan.predict_proba(X_test_bal)[:, 1]

    ctgan_results = [
        evaluate_scores(y_test_work, score_test_imb, threshold_ctgan, 'ctgan_xgboost_test_imbalanced'),
        evaluate_scores(y_test_bal, score_test_bal, threshold_ctgan, 'ctgan_xgboost_test_balanced'),
    ]

    for r in ctgan_results:
        print_eval(r)

    pd.DataFrame(ctgan_results).to_csv(OUT_DIR / 'ctgan_xgboost_results.csv', index=False)
    print('Tempo avaliação min:', (time.time() - start) / 60)
else:
    print('RUN_FULL_CTGAN=False')


## 15. Comparação com valores-alvo do artigo

In [ ]:
target = pd.DataFrame([
    {'label': 'paper_target_imbalanced', 'pr_auc': 0.93, 'roc_auc': 0.99, 'f1': 0.97},
])

frames = []
if Path(OUT_DIR / 'baseline_results.csv').exists():
    frames.append(pd.read_csv(OUT_DIR / 'baseline_results.csv'))
if Path(OUT_DIR / 'ctgan_xgboost_results.csv').exists():
    frames.append(pd.read_csv(OUT_DIR / 'ctgan_xgboost_results.csv'))

if frames:
    comparison = pd.concat(frames, ignore_index=True)
    display(comparison[['label', 'threshold', 'pr_auc', 'roc_auc', 'f1', 'precision', 'recall', 'tn', 'fp', 'fn', 'tp']])
    comparison.to_csv(OUT_DIR / 'all_results_comparison.csv', index=False)
else:
    print('Nenhum resultado encontrado ainda.')

display(target)


## 16. Curvas PR e ROC

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

if RUN_FULL_CTGAN:
    plt.figure(figsize=(7,5))
    PrecisionRecallDisplay.from_predictions(y_test_work, score_test_imb)
    plt.title('PR Curve - CTGAN + XGBoost - Test Imbalanced')
    plt.grid(True)
    plt.savefig(OUT_DIR / 'pr_curve_ctgan_imbalanced.png', dpi=160, bbox_inches='tight')
    plt.show()

    plt.figure(figsize=(7,5))
    RocCurveDisplay.from_predictions(y_test_work, score_test_imb)
    plt.title('ROC Curve - CTGAN + XGBoost - Test Imbalanced')
    plt.grid(True)
    plt.savefig(OUT_DIR / 'roc_curve_ctgan_imbalanced.png', dpi=160, bbox_inches='tight')
    plt.show()
else:
    print('RUN_FULL_CTGAN=False')


## 17. Salvar relatório JSON

In [ ]:
report = {
    'dataset': {
        'X_train_shape': tuple(X_train.shape),
        'X_test_shape': tuple(X_test.shape),
        'train_counts': train_counts,
        'test_counts': test_counts,
    },
    'params': {
        'xgboost': XGB_PARAMS,
        'ctgan': {
            'embedding_dim': CTGAN_EMBEDDING_DIM,
            'generator_dim': CTGAN_GENERATOR_DIM,
            'discriminator_dim': CTGAN_DISCRIMINATOR_DIM,
            'batch_size': CTGAN_BATCH_SIZE,
            'epochs': CTGAN_EPOCHS,
            'pac': CTGAN_PAC,
        },
        'quick_mode': QUICK_MODE,
        'run_full_ctgan': RUN_FULL_CTGAN,
    }
}

with open(OUT_DIR / 'experiment_report.json', 'w') as f:
    json.dump(report, f, indent=2, default=str)

print('Arquivos salvos em:', OUT_DIR)
!ls -lh /content/outputs_someip_xgboost


## Observações práticas

- Se o Colab reiniciar por falta de RAM, primeiro rode com `QUICK_MODE=True`.
- Para reprodução final, use Colab Pro/Pro+ ou máquina local com bastante RAM.
- Se CTGAN ficar inviável, rode primeiro o baseline sem CTGAN e depois reduza epochs/amostras para uma análise incremental.
- O threshold usado nas métricas finais deve ser escolhido no treino/validação, não no teste.
